In [3]:
from dotenv import load_dotenv
import os, json


load_dotenv(override=True)

openai_key = os.getenv("OPENAI_API_KEY")
openai_default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")
gemini_key = os.getenv("GOOGLE_API_KEY")
gemini_default_model = os.getenv("GEMINI_DEFAULT_MODEL", "gemini-4-hydrogen")
openai_embedding_model = os.getenv("EMBEDDING_MODEL")

In [4]:
##############################################
################ fallback 구조 ################
##############################################
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate


g_llm = init_chat_model(f'google_genai:{gemini_default_model}')
o_llm = init_chat_model(f'openai:{openai_default_model}')

with_fallbacks_llm = o_llm.with_fallbacks([g_llm])

In [5]:
# pdf lodaer
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
from langchain_openai import OpenAIEmbeddings


embeddings = OpenAIEmbeddings(model=openai_embedding_model)


def clean_tokenized_text(text: str) -> str:
    # 1. 모든 줄바꿈을 일단 공백 하나로 합침
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. 문단/항목 기호(◇, △) 앞에는 줄바꿈(\n\n)을 넣어 청킹 기준점 확보
    text = re.sub(r'\s*(◇|△)\s*', r'\n\n\1 ', text)
    
    # 3. 괄호 및 따옴표 안쪽의 불필요한 공백 제거: ( 서울 = 뉴스 1 ) -> (서울=뉴스1)
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    text = re.sub(r'\s*=\s*', '=', text)
    text = re.sub(r"'\s+([^']+?)\s+'", r"'\1'", text)
    text = re.sub(r'"\s+([^"]+?)\s+"', r'"\1"', text)
    
    # 4. 문장 부호(., ,) 앞의 불필요한 공백 제거
    text = re.sub(r'\s+([.,])', r'\1', text)
    
    # 5. 숫자와 단위/조사 사이 공백 제거 (예: 6 월 -> 6월, 23 명 -> 23명, 1 년 -> 1년)
    text = re.sub(r'(\d)\s+([가-힣])', r'\1\2', text)
    
    # 6. 한글 어간과 분리된 조사/어미 결합 (예: 한국 축구대표팀 이 -> 한국 축구대표팀이)
    # 자주 분리되는 한국어 조사/어미 패턴
    josa_pattern = r'(?<=[가-힣\)])\s+(은|는|이|가|을|를|의|에|에서|에게|로|으로|와|과|도|만|까지|부터|마저|조차|이나|나|이며|며|고|다고|라고|이라고|했다|됐다|밝혔다|말했다|설명했다|있다|없다|중|후|전|등|안|밖)(?=[\s.,\)\"\']|$)'
    # 여러 번 연달아 붙을 수 있으므로 2회 반복 적용 (예: 선수 측 의 -> 선수측의)
    for _ in range(2):
        text = re.sub(josa_pattern, r'\1', text)
        
    # 7. 문장 끝(. 또는 ." 또는 .') 뒤에 줄바꿈(\n\n)을 추가하여 RecursiveCharacterTextSplitter가 문장 단위로 자르기 좋게 만듦
    text = re.sub(r'(\.(?:["\'])?)\s+', r'\1\n\n', text)
    
    return text.strip()

manual_path = "data/2026 FIFA북중미 월드컵 아시아지역 2차 예선 6월 A대표팀 선수 소집 명단 발표.pdf"

pages = PyPDFLoader(manual_path).load()
# print(pages[0].page_content)
pages[0].page_content = clean_tokenized_text(pages[0].page_content)
# print(pages[0].page_content)


manual_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""] # \n\n(문단/문장) 기준으로 우선 분할
)

manual_chunks = manual_splitter.split_documents(pages)
print("청크 수:", len(manual_chunks))



청크 수: 7


In [6]:
from langchain_chroma import Chroma

soccer_store = Chroma.from_documents(
    documents=manual_chunks,
    embedding=embeddings,
    ids = [f'article-{i+1}' for i in range(len(manual_chunks))],
    persist_directory='soccer_chroma_db',
    collection_name='soccer_colletion'
)

In [7]:
soccer_retriever = soccer_store.as_retriever(
    search_type='mmr',
    search_kwargs={"k":5,
                   "score_threshold":0.3,
                   "fetch_k":5,
                   },
)

basic_prompt = ChatPromptTemplate.from_template(
    "다음 컨텍스트를 참고해서 질문에 답해줘.\n\n컨텍스트: {context}\n\n질문: {question}"
)


soccer_chain = {"context": soccer_retriever, "question":RunnablePassthrough()} | basic_prompt| o_llm | StrOutputParser()

questions = [
    "선발에서 제외된 선수 이름은?",
"소집 선수는 몇 명이니?",
"수비수로 뽑힌 선수는 모두 몇 명이고, 이름이 뭐니?"
]

for q in questions:
    print("Q:", q)
    print("A:", soccer_chain.invoke(q), "\n")

Q: 선발에서 제외된 선수 이름은?
A: 부상으로 제외된 선수는 김민재, 조규성, 설영우입니다. 

Q: 소집 선수는 몇 명이니?
A: 소집 선수는 총 23명입니다. 

Q: 수비수로 뽑힌 선수는 모두 몇 명이고, 이름이 뭐니?
A: 수비수로 뽑힌 선수는 **8명**입니다: 권경원, 박승욱, 조유민, 하창래, 김진수, 이명재, 황재원, 최준. 

